In [ ]:
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns 

In [ ]:
df = pd.read_csv("smartcart_customers.csv")
df.head()

## Step 1 : Data Preprocessing 

## Handling Missing Values 

In [ ]:
df["Income"] = df["Income"].fillna(df["Income"].median())

In [ ]:
df.isnull().sum()

## Feature Engineering

In [ ]:
# Creating a new feature age from the given feature Birth Year
df["Age"] = 2026 - df["Year_Birth"]

In [ ]:
df.head()

In [ ]:
# creating new feauture customer_tenure menas how much day has been till the date of cutomer join using customer joining date
df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"],dayfirst = True)
df["Customer_tenure"] = (df["Dt_Customer"].max() - df["Dt_Customer"]).dt.days

In [ ]:
df.head()

In [ ]:
# Creating new feature Total_spending by adding all given spending on different product
df["Total_Spending"] = df["MntWines"] + df["MntFruits"] + df["MntMeatProducts"] + df["MntFishProducts"] + df["MntSweetProducts"] + df["MntGoldProds"]

In [ ]:
df.head()

In [ ]:
df["Total_Children"] = df["Kidhome"] + df["Teenhome"]

In [ ]:
df.head()

In [ ]:
df["Education"] = df["Education"].replace({

    "Basic" : "Undergraduate","2n Cycle" : "Undergraduate",
    "Graduation" : "Graduate",
    "Master" : "Postgraduate", "PhD" : "Postgraduate"
})

In [ ]:
# Education 
df["Education"].value_counts()

In [ ]:
# Maritial Status
df["Living_Status"] = df["Marital_Status"].replace(
    {
        "Married" : "Partner" , "Together" : "Partner",
        "Single" : "Alone" , "Divorced" : "Alone" , "Widow" : "Alone",
        "Absurd" : "Alone" ,"YOLO" : "Alone"
    }
)

In [ ]:
df["Living_Status"].value_counts()

In [ ]:
cols = ["ID","Year_Birth","Marital_Status","Kidhome","Teenhome","Dt_Customer"]
spend_col = ["MntWines","MntFruits","MntMeatProducts","MntFishProducts","MntSweetProducts","MntGoldProds"]
cols_to_drop = cols + spend_col
df_cleaned  = df.drop(columns = cols_to_drop)

In [ ]:
df.columns

# Outlier

In [ ]:
cols = ["Income","Recency","Response","Age","Total_Spending","Total_Children"] 
sns.pairplot(df_cleaned [cols])

# Remove outliers 

In [ ]:
print("Size of data with outlier",len(df_cleaned))
df_cleaned = df_cleaned[df_cleaned["Age"]<90]
df_cleaned = df_cleaned[df_cleaned["Income"]<600_000]
print("Size of data without outlier",len(df_cleaned))

# Correlation of features 

In [ ]:
corr = df_cleaned.corr(numeric_only = True)

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(
    corr,
    annot = True,
    annot_kws={"size" : 6},
    cmap="coolwarm"
)

# Feature Encoding 

In [ ]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder()
cats_col = ["Education","Living_Status"]
en_cols = ohe.fit_transform(df_cleaned[cats_col])

In [ ]:
en_df = pd.DataFrame(en_cols.toarray(),columns=ohe.get_feature_names_out(cats_col),index = df_cleaned.index)

In [ ]:
df_encoded = pd.concat([df_cleaned.drop(columns = cats_col),en_df],axis = 1)
df_encoded.head()

# Scaling 

In [ ]:
from sklearn.preprocessing import StandardScaler 

x = df_encoded
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)
x_scaled

# Visiualization 

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components = 3)
x_pca = pca.fit_transform(x_scaled)

In [ ]:
pca.explained_variance_ratio_

In [ ]:
fig = plt.figure(figsize=(8,6))
ax = fig.add_subplot(111,projection = "3d")
ax.scatter(x_pca[:,0],x_pca[:,1],x_pca[:,2])
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_zlabel("PC3")


# Analyse K value 
## 1. Elbow Method

In [ ]:
from sklearn.cluster import KMeans
from kneed import KneeLocator
wcss = []

for k in range(1,11):
    kmeans = KMeans(n_clusters = k,random_state=42)
    labels = kmeans.fit_predict(x_pca)
    v = kmeans.inertia_
    wcss.append(v)

In [ ]:
knee = KneeLocator(range(1,11),wcss,curve="convex",direction="decreasing")
optimal_k = knee.elbow
optimal_k

# Clustering 

In [ ]:
# KMeans 
kmeans = KMeans(n_clusters = 4,random_state =42)
label = kmeans.fit_predict(x_pca)
fig = plt.figure(figsize=(8,6))
ax = fig.add_subplot(111,projection = "3d")
ax.scatter(x_pca[:,0],x_pca[:,1],x_pca[:,2],c=label)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_zlabel("PC3")


In [ ]:
# Heirarchical Clustering 
from sklearn.cluster import AgglomerativeClustering 

agg_clf = AgglomerativeClustering(n_clusters = 4,linkage = "ward")
labels = agg_clf.fit_predict(x_pca)
fig = plt.figure(figsize=(8,6))
ax = fig.add_subplot(111,projection = "3d")
ax.scatter(x_pca[:,0],x_pca[:,1],x_pca[:,2],c=labels)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_zlabel("PC3")

# Characterization of Clusters
## what kind of info each cluster have 

In [ ]:
df_cleaned.drop("Labels", axis=1)
df_cleaned["Clusters"] = labels

In [ ]:
df_cleaned.head()

In [ ]:
pal = ["red","green","blue","yellow"]
sns.countplot(x=df_cleaned["Clusters"],palette = pal , hue = df_cleaned["Clusters"])

In [ ]:
sns.scatterplot(x=df_cleaned["Total_Spending"],y=df_cleaned["Income"],hue=df_cleaned["Clusters"],palette=pal)

In [ ]:
# from this we can find valuble data from the clusters 
x["Clusters"] =  labels
x.head()
cluster_summary = x.groupby("Clusters").mean()
print(cluster_summary)